In [3]:
import pandas as pd
import numpy as ny
import pyodbc as pys
import re
from datetime import datetime
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [4]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
             
             select NUM_IDENTIFICACION,CANT_MATERIAS, CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS, CICLO, COD_UNIDAD,
                CASE WHEN COD_PENSUM LIKE '%D' THEN REPLACE (COD_PENSUM ,'D','')
                WHEN COD_PENSUM LIKE '%N' THEN REPLACE (COD_PENSUM ,'N','')
                ELSE COD_PENSUM END AS PENSUM_L 
      
            from openquery([172.16.1.175],
            'select * from iceberg.CICLOS_X_ESTUDIANTE')
                      
        
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

### Reconocimiento de titulo

In [5]:
df[df['NUM_IDENTIFICACION']=='1018477214']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
470950,1018477214,23.0,23.0,23.0,TECNOLOGO,VP15G,VG002
470951,1018477214,20.0,17.0,0.0,PROFESIONAL,VP15G,VG002
470952,1018477214,29.0,29.0,29.0,TECNICO,VP15G,VG002


### Reconocimiento de tecnico

In [6]:
df[df['NUM_IDENTIFICACION']=='1234196804']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
1559603,1234196804,15.0,7.0,0.0,TECNOLOGO,VGM62,3040
1559604,1234196804,23.0,0.0,NaN,PROFESIONAL,VGM62,3040
1559605,1234196804,29.0,29.0,29.0,TECNICO,VGM62,3040


### Homologante

In [7]:
df[df['NUM_IDENTIFICACION']=='16935089']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
1599383,16935089,17.0,3.0,3.0,TECNOLOGO,VTC63,5016
1599384,16935089,21.0,1.0,1.0,PROFESIONAL,VTC63,5016
1599385,16935089,30.0,26.0,18.0,TECNICO,VTC63,5016


#### Normal

In [8]:
df[df['NUM_IDENTIFICACION']=='50876215']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L
1760069,50876215,18.0,0.0,NaN,TECNOLOGO,VTM14,6017
1760070,50876215,29.0,0.0,NaN,PROFESIONAL,VTM14,6017
1760071,50876215,25.0,7.0,0.0,TECNICO,VTM14,6017


In [9]:
def asignar_marca(row):
    if pd.isna(row['CANT_MAT_HOMOLOGADAS']) or row['CANT_MAT_HOMOLOGADAS'] == 0:
        return 'Normal'
    elif row['CANT_MATERIAS'] == row['CANT_MAT_HOMOLOGADAS']:
        return 'Reconocimiento'
    else:
        return 'Homologante'

df['Marca'] = df.apply(asignar_marca, axis=1)

In [10]:
len(df[df['NUM_IDENTIFICACION']=='83183336'])

21

In [11]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT COD_PROG_TECNICO,COD_PROG_TECNOLOGO,COD_PROG_PROFESIONAL,PENSUM 
		    FROM CUN.MALLASLAURA 
      
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_mallas_l = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [12]:
df_a = pd.merge(df, df_mallas_l, how='left', 
                left_on=['COD_UNIDAD', 'PENSUM_L'], 
                right_on=['COD_PROG_TECNICO', 'PENSUM'])

df_a = pd.merge(df_a, df_mallas_l, how='left', 
                left_on=['COD_UNIDAD', 'PENSUM_L'], 
                right_on=['COD_PROG_TECNOLOGO', 'PENSUM'])

df_a = pd.merge(df_a, df_mallas_l, how='left', 
                left_on=['COD_UNIDAD', 'PENSUM_L'], 
                right_on=['COD_PROG_PROFESIONAL', 'PENSUM'])

In [13]:
import numpy as np

def obtener_codigo(row):
    if row['CICLO'] == 'TECNICO':
        return next((x for x in [row['COD_PROG_TECNICO_x'], row['COD_PROG_TECNICO_y'], row['COD_PROG_TECNICO']] if pd.notna(x)), np.nan)
    elif row['CICLO'] == 'TECNOLOGO':
        return next((x for x in [row['COD_PROG_TECNOLOGO_x'], row['COD_PROG_TECNOLOGO_y'], row['COD_PROG_TECNOLOGO']] if pd.notna(x)), np.nan)
    elif row['CICLO'] == 'PROFESIONAL':
        return next((x for x in [row['COD_PROG_PROFESIONAL_x'], row['COD_PROG_PROFESIONAL_y'], row['COD_PROG_PROFESIONAL']] if pd.notna(x)), np.nan)
    return np.nan

df_a['COD_PROG_FINAL'] = df_a.apply(obtener_codigo, axis=1)

In [14]:
df_a = df_a[['NUM_IDENTIFICACION', 'CANT_MATERIAS', 'CANT_MAT_APRO',
       'CANT_MAT_HOMOLOGADAS', 'CICLO', 'COD_UNIDAD', 'PENSUM_L', 'Marca','COD_PROG_FINAL']]

In [15]:
df_a[df_a['NUM_IDENTIFICACION']=='1018477214']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L,Marca,COD_PROG_FINAL
470950,1018477214,23.0,23.0,23.0,TECNOLOGO,VP15G,VG002,Reconocimiento,VG15G
470951,1018477214,20.0,17.0,0.0,PROFESIONAL,VP15G,VG002,Normal,VP15G
470952,1018477214,29.0,29.0,29.0,TECNICO,VP15G,VG002,Reconocimiento,VT15G


In [16]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT Identificacion,ciclo_sugerido,Cod_unidad_sug,pensum_limpio,Estado_Alumno 
            FROM DESERCION.Python_CUN_Simulado_cambios_ciclo
            where Estado_Alumno = 'Traslado'
                  

    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_rt = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [17]:
df_a.shape

(2265450, 9)

In [18]:
df_b = pd.merge(df_a,df_rt,how='left',left_on=['NUM_IDENTIFICACION','COD_PROG_FINAL','PENSUM_L'],
                right_on=['Identificacion','Cod_unidad_sug','pensum_limpio'])

In [19]:
df_b = df_b.drop_duplicates()

In [20]:
# Identificar los estudiantes que tienen 'Traslado' en algún ciclo, agrupando por NUM_IDENTIFICACION y CICLO
traslados = df_b[df_b['Estado_Alumno'] == 'Traslado'][['NUM_IDENTIFICACION', 'CICLO']].drop_duplicates()

# Crear una máscara con todas las combinaciones de NUM_IDENTIFICACION y CICLO que deben ser Traslado
mask = df_b.set_index(['NUM_IDENTIFICACION', 'CICLO']).index.isin(traslados.set_index(['NUM_IDENTIFICACION', 'CICLO']).index)

# Actualizar directamente todas las filas que cumplen la condición
df_b.loc[mask, 'Estado_Alumno'] = 'Traslado'


In [21]:
# df_b.loc[(df_b['Marca'].isin(['Reconocimiento','Homologante']))&
#          (df_b['Estado_Alumno']=='Traslado'),
#          'Marca'] = df_b['Estado_Alumno']

In [22]:
df_b[df_b['NUM_IDENTIFICACION']=='1083026192']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_UNIDAD,PENSUM_L,Marca,COD_PROG_FINAL,Identificacion,ciclo_sugerido,Cod_unidad_sug,pensum_limpio,Estado_Alumno
1153350,1083026192,19.0,19.0,19.0,TECNOLOGO,1GR02,1030,Reconocimiento,1GR02,NaN,NaN,NaN,NaN,NaN
1153351,1083026192,30.0,10.0,10.0,PROFESIONAL,1GR02,1030,Homologante,1PR02,NaN,NaN,NaN,NaN,NaN
1153352,1083026192,26.0,26.0,26.0,TECNICO,1GR02,1030,Reconocimiento,1TR02,1083026192,TECNICO,1TR02,1030,Traslado
1153353,1083026192,19.0,0.0,NaN,TECNOLOGO,1TR02,1030,Normal,1GR02,NaN,NaN,NaN,NaN,NaN
1153354,1083026192,30.0,0.0,NaN,PROFESIONAL,1TR02,1030,Normal,1PR02,NaN,NaN,NaN,NaN,NaN
1153355,1083026192,26.0,21.0,8.0,TECNICO,1TR02,1030,Homologante,1TR02,1083026192,TECNICO,1TR02,1030,Traslado
1153356,1083026192,19.0,19.0,0.0,TECNOLOGO,1TR02,1030,Normal,1GR02,NaN,NaN,NaN,NaN,NaN
1153357,1083026192,30.0,10.0,0.0,PROFESIONAL,1TR02,1030,Normal,1PR02,NaN,NaN,NaN,NaN,NaN
1153358,1083026192,26.0,26.0,21.0,TECNICO,1TR02,1030,Homologante,1TR02,1083026192,TECNICO,1TR02,1030,Traslado


In [23]:
df_final = df_b[['NUM_IDENTIFICACION', 'CANT_MATERIAS', 'CANT_MAT_APRO',
       'CANT_MAT_HOMOLOGADAS', 'CICLO','COD_PROG_FINAL','PENSUM_L',
       'Marca', 'Estado_Alumno']]

In [24]:
df_final['Estado_Alumno'].unique()
df_final[df_final['Estado_Alumno']=='Traslado']
df_final[df_final['NUM_IDENTIFICACION']=='1083026192']
df_final['Marca'].unique()

array(['Normal', 'Reconocimiento', 'Homologante'], dtype=object)

In [25]:
df_final2 = df_final

In [26]:
# Asignar una prioridad numérica para facilitar el ordenamiento
prioridad = {'Normal': 1, 'Homologante': 2, 'Reconocimiento': 3}
df_final2['prioridad'] = df_final2['Marca'].map(prioridad)

# Ordenar el DataFrame por cedula, ciclo, cod_unidad, pensum y prioridad
df_final2 = df_final2.sort_values(['NUM_IDENTIFICACION', 'CICLO', 'COD_PROG_FINAL', 'PENSUM_L', 'prioridad'])

# Eliminar duplicados dejando el primer registro (que será el de mayor prioridad)
df_final2 = df_final2.drop_duplicates(subset=['NUM_IDENTIFICACION', 'CICLO', 'COD_PROG_FINAL', 'PENSUM_L'], keep='first')

# Eliminar la columna de prioridad ya que solo fue utilizada para ordenar
df_final2 = df_final2.drop(columns=['prioridad'])

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_34424\4216554024.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final2['prioridad'] = df_final2['Marca'].map(prioridad)


In [27]:
df_final2.shape

(1708059, 9)

In [28]:
df_final2[df_final2['NUM_IDENTIFICACION']=='1006006760']

,NUM_IDENTIFICACION,CANT_MATERIAS,CANT_MAT_APRO,CANT_MAT_HOMOLOGADAS,CICLO,COD_PROG_FINAL,PENSUM_L,Marca,Estado_Alumno
179636,1006006760,20.0,15.0,0.0,PROFESIONAL,1PS58,2033,Normal,NaN
179638,1006006760,0.0,0.0,NaN,PROFESIONAL,NaN,CE421,Normal,NaN
179637,1006006760,29.0,29.0,9.0,TECNICO,1TS58,2033,Homologante,NaN
179639,1006006760,0.0,0.0,NaN,TECNICO,NaN,CE421,Normal,NaN
179635,1006006760,23.0,22.0,0.0,TECNOLOGO,1GS58,2033,Normal,NaN
179640,1006006760,0.0,0.0,NaN,TECNOLOGO,NaN,CE421,Normal,NaN


In [29]:
##df_final2.to_csv('Reconocimiento.csv',index=False)

In [30]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'Reconocimiento'
schema_name = 'SRUBIO'

# Send the data to sql server
df_final2.to_sql(
    name=table_name,
    schema=schema_name, 
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()